In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr
import matplotlib.pyplot as plt

from langchain_core.documents import Document
from langchain_teradata import TeradataVectorStore       # <--- vector store
import boto3                                             # <--- to enable embedding models
from langchain_aws import BedrockEmbeddings              # <--- to enable embedding models
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

import teradataml as tdml
from teradatagenai import TeradataAI

# Load API key safely from .env
load_dotenv(dotenv_path='../../.env')

region      = "us-east-1"
bedrock_url = f"https://bedrock-runtime.{region}.amazonaws.com/openai/v1"

import json

from more information got to : Teradata-Package-for-LangChain-Function-Reference[https://docs.teradata.com/r/Enterprise/Teradata-Package-for-LangChain-Function-Reference]

## 0. Connection to Teradata Vantage

In [ ]:
tdml.create_context(host=os.getenv('VANTAGE_HOST'), username=os.getenv('VANTAGE_USER'), password = os.getenv('VANTAGE_PASSWORD'), encryptdata=True)

In [ ]:
with open("/home/denis/notebooks/auth_token.json", "r") as f:
    auth_token = json.load(f)

tdml.set_auth_token(**auth_token)

## 1. Documents for the demo

In [ ]:
docs = [
    Document(page_content="LangChain is a framework for building LLM applications."),
    Document(page_content="RAG means Retrieval-Augmented Generation."),
    Document(page_content="BM25 is a keyword-based retrieval algorithm."),
    Document(page_content="Vector databases are useful for semantic search."),
    Document(page_content="Paris is the capital of France."),
    Document(page_content="Denis Molin is a great guy."),
]

## 2. Retriever / vector store

In [ ]:
embeddings = TeradataAI(
    api_type   = "aws",
    model_name = "amazon.titan-embed-text-v2:0",
    access_key = os.getenv('AWS_ACCESS_KEY'),
    region     = region
)

In [ ]:
vectorstore = TeradataVectorStore( name       = 'dmvectorstore')

In [ ]:
import pandas as pd
import numpy as np
if isinstance(vectorstore.status(), pd.core.frame.DataFrame):
    vectorstore.destroy()

In [ ]:
vectorstore = TeradataVectorStore.from_documents(
    name       = 'dmvectorstore',
    documents  = docs,
    embedding  = embeddings,
)

## 3. LLM

In [ ]:
llm = ChatOpenAI(
    model       = "mistral.ministral-3-14b-instruct",
    api_key     = os.getenv("AWS_BEARER_TOKEN_BEDROCK"),
    base_url    = bedrock_url,
    temperature = 0
)
llm.invoke("Explain RAG in one sentence")

## 4. RAG function

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """
You are a helpful assistant answering questions using only the provided context.

Rules:
- If the context contains relevant information, answer using that information, even if the wording of the question is indirect.
- You may restate or lightly infer from the context, but do not invent facts not supported by it.
- If the context does not contain enough information to answer, say:
  "I don't know based on the provided documents."
- When possible, answer directly and briefly.
"""

def build_rag_answer(vectorstore, llm, k=4, return_sources=False):
    #retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    retriever = vectorstore.as_retriever(
        search_type="similarity", #"mmr",
        search_kwargs={"top_k":4}
    )

    prompt = ChatPromptTemplate.from_template("""
<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

Answer the question using only the context below.

If the context contains partial but relevant information, use it to give the best grounded answer.
If the context does not contain enough information, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}
[/INST]
""")

    def rag_answer(question: str):
        docs = retriever.invoke(question)
        context = "\n\n".join(doc.page_content for doc in docs)

        messages = prompt.format_messages(
            system_prompt = SYSTEM_PROMPT,
            context       = context,
            question      = question,
        )

        response = llm.invoke(messages)

        if hasattr(response, "content"):
            answer = str(response.content)
        else:
            answer = str(response)

        if not return_sources:
            return answer

        sources_md = "\n\n".join(
            f"**Source {i+1}**\n\n{doc.page_content}"
            for i, doc in enumerate(docs)
        )

        if not sources_md.strip():
            sources_md = "No sources retrieved."

        return answer, sources_md

    return rag_answer

## 5. Chat interface

In [ ]:
from rag_ui import launch_rag_ui

# Example:
# vectorstore = FAISS.from_documents(chunks, embedding_model)
# llm = ChatOpenAI(model="gpt-4o-mini")  # or your Mistral / LiteLLM wrapper

rag_answer = build_rag_answer(vectorstore=vectorstore, llm=llm, k=4,return_sources=True)

def chat(message, history):
    answer, sources_md = rag_answer(message)
    history = history + [{"role": "user", "content": message},
                         {"role": "assistant", "content": answer}]
    return history, sources_md, ""

demo = launch_rag_ui(
    chat_fn=chat,
    title="Simple RAG Demo (Vector Embeddings)",
    port=7860,
)

## 6. Embedding vectors inspection

### 6.1 Inspect the vector store content

In [ ]:
import numpy as np

details            = vectorstore.get_details()
indexes_embeddings = vectorstore.get_indexes_embeddings()

In [ ]:
details

In [ ]:
indexes_embeddings

In [ ]:
value = indexes_embeddings.to_pandas(num_rows=1).vector_index.values[0]
np.frombuffer(value, dtype=np.float32).shape

In [ ]:
num_vectors = indexes_embeddings.shape[0]
vector_dim  = np.frombuffer(indexes_embeddings.to_pandas(num_rows=1).vector_index.values[0],dtype=np.float32).shape[0] if num_vectors > 0 else 0

print(f"\nNumber of demo vectors: {num_vectors}")
print(f"Vector dimension: {vector_dim}")

### 6.2 Prepare labels

In [ ]:
vector_data = indexes_embeddings.to_pandas()
vector_data['vector_index']            = vector_data['vector_index'].apply(lambda x:np.frombuffer(x, dtype=np.float64))
vector_data['vector_index_normalized'] = vector_data['vector_index_normalized'].apply(lambda x:np.frombuffer(x, dtype=np.float64))
vector_data

In [ ]:
labels = [doc[:40].replace("\n", " ") for doc in vector_data.text.values]


### 6.3 Compare two query embeddings

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

query_1 = vector_data.text.values[0]
query_2 = vector_data.text.values[1]

q1_vec = vector_data.vector_index.values[0].reshape(1, -1)
q2_vec = vector_data.vector_index.values[1].reshape(1, -1)

query_similarity = cosine_similarity(q1_vec, q2_vec)[0][0]
print(f'Cosine similarity between "{query_1}" and "{query_2}": {query_similarity:.4f}')

### 6.4 Project document + query vectors into 2D with PCA

In [ ]:
doc_vectors = vector_data.vector_index.tolist()
all_vectors = np.vstack([doc_vectors, q1_vec, q2_vec])

pca         = PCA(n_components=2)
all_2d      = pca.fit_transform(all_vectors)

doc_2d      = all_2d[:num_vectors]
q1_2d       = all_2d[num_vectors]
q2_2d       = all_2d[num_vectors + 1]

### 6.5 Matplotlib visualization

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 7))

# Document points
plt.scatter(doc_2d[:, 0], doc_2d[:, 1], s=80, label="Documents")

# Faint arrows from origin to document vectors
for i, label in enumerate(labels):
    x, y = doc_2d[i]
    plt.arrow(
        0, 0, x, y,
        alpha=0.25,
        head_width=0.03,
        length_includes_head=True
    )
    plt.text(x, y, label, fontsize=9)

# Query points
plt.scatter(q1_2d[0], q1_2d[1], s=140, marker="X", label="Query 1")
plt.scatter(q2_2d[0], q2_2d[1], s=140, marker="X", label="Query 2")

# Query arrows
plt.arrow(
    0, 0, q1_2d[0], q1_2d[1],
    alpha=0.7,
    head_width=0.04,
    length_includes_head=True
)
plt.arrow(
    0, 0, q2_2d[0], q2_2d[1],
    alpha=0.7,
    head_width=0.04,
    length_includes_head=True
)

plt.text(q1_2d[0], q1_2d[1], f" {query_1}", fontsize=10)
plt.text(q2_2d[0], q2_2d[1], f" {query_2}", fontsize=10)

plt.axhline(0, linewidth=0.5)
plt.axvline(0, linewidth=0.5)

plt.title("Document and Query Embeddings (PCA projection)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.show()

### 6.6 Interactive Plotly visualization

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd

pio.renderers.default = "notebook_connected"

plot_df = pd.DataFrame({
    "x": np.concatenate([doc_2d[:, 0], [q1_2d[0], q2_2d[0]]]),
    "y": np.concatenate([doc_2d[:, 1], [q1_2d[1], q2_2d[1]]]),
    "label": labels + [query_1, query_2],
    "type": ["Document"] * num_vectors + ["Query", "Query"],
})

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    text="label",
    color="type",
    title="Interactive PCA Projection of Embeddings",
    hover_data={"x": ':.3f', "y": ':.3f', "label": True, "type": True},
)

fig.update_traces(textposition="top center")
fig.update_layout(width=900, height=650)
fig.show()

### 6.7 Retrieve top matching documents

We convert the distance into a similarity score using:

$similarity = \frac{1}{1 + distance}$

In [ ]:
query = "What is RAG?"

results = vectorstore.similarity_search(
    question    = query,
    top_k       = 3,
    return_type = "json",
)

print(f"Top matches for: {query}\n")
print("Raw similarity_search response:")
print(results)

print("\nParsed matches:\n")

if isinstance(results, list):
    for i, row in enumerate(results, 1):
        print(f"Result {i}")
        if isinstance(row, Document):
            print(row.page_content)
        elif isinstance(row, dict):
            for key in ("score", "similarity", "distance", "content", "page_content", "text", "document"):
                if key in row:
                    print(f"{key} = {row[key]}")
            metadata = row.get("metadata")
            if metadata:
                print("metadata =", metadata)
        else:
            print(row)
        print("-" * 60)
else:
    print("Unexpected response type:", type(results))
    print(results)